In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/raw/fire_incident_data.csv")

df_clean = df.copy()

print("Original dataset:", df.shape)
print("Cleaning dataset:", df_clean.shape)

Original dataset: (36564, 43)
Cleaning dataset: (36564, 43)


C:\Users\BSE\AppData\Local\Temp\ipykernel_14864\416099318.py:6: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/fire_incident_data.csv")


convert all column names to lowercase with underscores for consistent naming


In [3]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print(df_clean.columns.tolist())

['_id', 'incident_number', 'initial_cad_event_type', 'final_incident_type', 'exposures', 'incident_station_area', 'incident_ward', 'intersection', 'latitude', 'longitude', 'property_use', 'building_status', 'tfs_alarm_time', 'tfs_arrival_time', 'ext_agent_app_or_defer_time', 'fire_under_control_time', 'last_tfs_unit_clear_time', 'number_of_responding_apparatus', 'number_of_responding_personnel', 'count_of_persons_rescued', 'tfs_firefighter_casualties', 'civilian_casualties', 'estimated_number_of_persons_displaced', 'estimated_dollar_loss', 'business_impact', 'status_of_fire_on_arrival', 'method_of_fire_control', 'area_of_origin', 'level_of_origin', 'extent_of_fire', 'smoke_spread', 'ignition_source', 'material_first_ignited', 'possible_cause', 'smoke_alarm_at_fire_origin', 'smoke_alarm_at_fire_origin_alarm_failure', 'smoke_alarm_at_fire_origin_alarm_type', 'smoke_alarm_impact_on_persons_evacuating_impact_on_evacuation', 'fire_alarm_system_presence', 'fire_alarm_system_operation', 'fire

convert date and time columns to datetime to facilitate the calculations that will use this columns

In [4]:
time_columns = [
    "tfs_alarm_time",
    "tfs_arrival_time",
    "fire_under_control_time",
    "last_tfs_unit_clear_time"
]

for col in time_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

print(df_clean[time_columns].dtypes)

display(df_clean[time_columns].head())

tfs_alarm_time              datetime64[ns]
tfs_arrival_time            datetime64[ns]
fire_under_control_time     datetime64[ns]
last_tfs_unit_clear_time    datetime64[ns]
dtype: object


,tfs_alarm_time,tfs_arrival_time,fire_under_control_time,last_tfs_unit_clear_time
0,2018-02-24 21:04:29,2018-02-24 21:10:11,2018-02-24 21:15:40,2018-02-24 21:38:31
1,2018-02-24 21:24:43,2018-02-24 21:29:31,2018-02-24 21:32:24,2018-02-24 21:35:58
2,2018-02-25 13:29:59,2018-02-25 13:36:49,NaT,2018-02-25 14:14:03
3,2018-02-25 14:13:39,2018-02-25 14:18:07,2018-02-25 14:20:00,2018-02-25 15:07:42
4,2018-02-25 18:20:43,2018-02-25 18:26:19,NaT,2018-02-25 18:34:24


check if there are missing dates after the conversion

In [6]:
# Check missing values in datetime columns
datetime_missing = df_clean[time_columns].isnull().sum()

print("Missing values after datetime conversion:")
print(datetime_missing)

Missing values after datetime conversion:
tfs_alarm_time                  0
tfs_arrival_time                1
fire_under_control_time     10539
last_tfs_unit_clear_time        1
dtype: int64


Standardize missing text values(making sure that missing information is represented consistently in our dataset.)

In [7]:
# Identify text columns
text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

# Count common missing-value placeholders
placeholders = ["", "N/A", "NA", "NULL", "None", "Unknown"]

for col in text_columns:
    counts = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .value_counts()
    )

    matches = counts[
        counts.index.str.upper().isin(
            [p.upper() for p in placeholders]
        )
    ]

    if not matches.empty:
        print(f"\n{col}")
        print(matches)


incident_station_area
incident_station_area
    2
Name: count, dtype: Int64

intersection
intersection
    2
Name: count, dtype: Int64


Check missing-value percentages after conversion

In [8]:
missing_summary = pd.DataFrame({
    "Missing_Count": df_clean.isna().sum(),
    "Missing_Percentage": (
        df_clean.isna().mean() * 100
    ).round(2)
})

display(
    missing_summary[
        missing_summary["Missing_Count"] > 0
    ].sort_values(
        "Missing_Percentage",
        ascending=False
    )
)

,Missing_Count,Missing_Percentage
exposures,25418,69.52
sprinkler_system_operation,18486,50.56
smoke_alarm_at_fire_origin_alarm_failure,18486,50.56
business_impact,18487,50.56
estimated_number_of_persons_displaced,18485,50.56
smoke_spread,18487,50.56
extent_of_fire,18487,50.56
smoke_alarm_at_fire_origin,18486,50.56
fire_alarm_system_presence,18486,50.56
smoke_alarm_at_fire_origin_alarm_type,18486,50.56


Check negative values in important numeric columns

In [10]:
numeric_columns = [
    "estimated_dollar_loss",
    "civilian_casualties",
    "tfs_firefighter_casualties",
    "count_of_persons_rescued",
    "number_of_responding_apparatus",
    "number_of_responding_personnel"
]

for col in numeric_columns:
    negative_count = (df_clean[col] < 0).sum()
    print(f"{col}: {negative_count} negative values")

estimated_dollar_loss: 0 negative values
civilian_casualties: 0 negative values
tfs_firefighter_casualties: 0 negative values
count_of_persons_rescued: 0 negative values
number_of_responding_apparatus: 0 negative values
number_of_responding_personnel: 0 negative values


Replace blank strings with NaN

In [11]:
# Replace empty strings and whitespace-only values with NaN
text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

df_clean[text_columns] = df_clean[text_columns].replace(
    r"^\s*$",
    np.nan,
    regex=True
)

# Verify the results
print("Missing station areas:",
      df_clean["incident_station_area"].isna().sum())

print("Missing intersections:",
      df_clean["intersection"].isna().sum())

Missing station areas: 3
Missing intersections: 4


Check negative values in all numerical columns

In [12]:
# Select all numerical columns
numeric_cols = df_clean.select_dtypes(
    include=np.number
).columns

# Count negative values
negative_counts = (df_clean[numeric_cols] < 0).sum()

# Display columns containing negative values
display(negative_counts[negative_counts > 0])

longitude    36559
dtype: int64

Negative logitude is normal.

validate longitude and latitude

In [13]:
print("Latitude range:")
print(df_clean["latitude"].min(), "to", df_clean["latitude"].max())

print("\nLongitude range:")
print(df_clean["longitude"].min(), "to", df_clean["longitude"].max())

Latitude range:
0.0 to 43.8554002362993

Longitude range:
-79.63946 to 0.0


count zero cordinates

In [14]:
print("Zero latitude:",
      (df_clean["latitude"] == 0).sum())

print("Zero longitude:",
      (df_clean["longitude"] == 0).sum())

# Display affected records
invalid_coordinates = df_clean[
    (df_clean["latitude"] == 0) |
    (df_clean["longitude"] == 0)
]

display(
    invalid_coordinates[
        ["incident_number", "latitude", "longitude", "intersection"]
    ]
)

Zero latitude: 3
Zero longitude: 3


,incident_number,latitude,longitude,intersection
30875,F22160636,0.0,0.0,YONGE ST/STEELES AVE E NY
35777,F24131551,0.0,0.0,VICTORIA PARK AVE/SHEPPARD AVE E SC
35837,F24010559,0.0,0.0,VICTORIA PARK AVE/SHEPPARD AVE E SC


replace invalid coordinates with NAN

In [15]:
# Identify records with zero coordinates
invalid_mask = (
    (df_clean["latitude"] == 0) |
    (df_clean["longitude"] == 0)
)

# Replace invalid coordinates with NaN
df_clean.loc[
    invalid_mask,
    ["latitude", "longitude"]
] = np.nan

# Verify the correction
print("Zero latitude:", (df_clean["latitude"] == 0).sum())
print("Zero longitude:", (df_clean["longitude"] == 0).sum())

print("Missing latitude:", df_clean["latitude"].isna().sum())
print("Missing longitude:", df_clean["longitude"].isna().sum())

Zero latitude: 0
Zero longitude: 0
Missing latitude: 5
Missing longitude: 5


investigate numerical outliers

In [16]:
columns_to_check = [
    "estimated_dollar_loss",
    "civilian_casualties",
    "tfs_firefighter_casualties",
    "count_of_persons_rescued",
    "number_of_responding_apparatus",
    "number_of_responding_personnel"
]

display(
    df_clean[columns_to_check].describe().T
)

,count,mean,std,min,25%,50%,75%,max
estimated_dollar_loss,26649.0,37466.138392,380092.220615,0.0,300.0,3000.0,20000.0,50000000.0
civilian_casualties,25867.0,0.072718,0.357953,0.0,0.0,0.0,0.0,15.0
tfs_firefighter_casualties,36342.0,0.009521,0.116202,0.0,0.0,0.0,0.0,4.0
count_of_persons_rescued,36341.0,0.039294,0.938504,0.0,0.0,0.0,0.0,86.0
number_of_responding_apparatus,36342.0,6.288289,7.446613,0.0,1.0,5.0,8.0,436.0
number_of_responding_personnel,36342.0,21.118100,23.437317,0.0,4.0,18.0,26.0,1275.0


inspect the incidents with the highest values

In [17]:
# Investigate the top 5 highest values in each column

columns_to_check = [
    "estimated_dollar_loss",
    "civilian_casualties",
    "tfs_firefighter_casualties",
    "count_of_persons_rescued",
    "number_of_responding_apparatus",
    "number_of_responding_personnel"
]

for col in columns_to_check:

    print(f"\nTOP 5 HIGHEST VALUES: {col}")

    display(
        df_clean.nlargest(5, col)[
            [
                "incident_number",
                "final_incident_type",
                col
            ]
        ]
    )


TOP 5 HIGHEST VALUES: estimated_dollar_loss


,incident_number,final_incident_type,estimated_dollar_loss
3364,F19046467,01 - Fire,50000000.0
23756,F21104931,01 - Fire,15000000.0
15167,F11031347,01 - Fire,13000000.0
14862,F17013467,01 - Fire,10000000.0
14879,F17014451,01 - Fire,10000000.0



TOP 5 HIGHEST VALUES: civilian_casualties


,incident_number,final_incident_type,civilian_casualties
14494,F16011041,01 - Fire,15.0
950,F18071851,01 - Fire,8.0
18056,F14049474,01 - Fire,8.0
16382,F15066203,01 - Fire,7.0
15692,F15029484,01 - Fire,6.0



TOP 5 HIGHEST VALUES: tfs_firefighter_casualties


,incident_number,final_incident_type,tfs_firefighter_casualties
14999,F15080030,01 - Fire,4.0
1511,F19017429,"02 - Explosion (including during Fire, excludi...",3.0
14851,F16094934,01 - Fire,3.0
14872,F15046633,01 - Fire,3.0
14977,F12003343,01 - Fire,3.0


... (2 outputs left out)


TOP 5 HIGHEST VALUES: number_of_responding_apparatus


,incident_number,final_incident_type,number_of_responding_apparatus
14977,F12003343,01 - Fire,436.0
14770,F14027175,01 - Fire,182.0
4540,F18089480,01 - Fire,175.0
3364,F19046467,01 - Fire,167.0
14879,F17014451,01 - Fire,167.0



TOP 5 HIGHEST VALUES: number_of_responding_personnel


,incident_number,final_incident_type,number_of_responding_personnel
14977,F12003343,01 - Fire,1275.0
14770,F14027175,01 - Fire,594.0
4540,F18089480,01 - Fire,537.0
14879,F17014451,01 - Fire,520.0
3364,F19046467,01 - Fire,515.0


inspect selected incidents

In [18]:
# Incidents requiring closer investigation
suspicious_incidents = [
    "F24081362",
    "F18090586",
    "F12003343",
    "F19046467"
]

# Display their relevant information
columns_to_display = [
    "incident_number",
    "final_incident_type",
    "estimated_dollar_loss",
    "civilian_casualties",
    "count_of_persons_rescued",
    "number_of_responding_apparatus",
    "number_of_responding_personnel",
    "property_use",
    "intersection"
]

display(
    df_clean.loc[
        df_clean["incident_number"].isin(suspicious_incidents),
        columns_to_display
    ].set_index("incident_number").T
)

incident_number,F24081362,F19046467,F18090586,F12003343
final_incident_type,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand...",01 - Fire,01 - Fire,01 - Fire
estimated_dollar_loss,0.0,50000000.0,3000.0,1000000.0
civilian_casualties,NaN,0.0,0.0,0.0
count_of_persons_rescued,86.0,0.0,86.0,0.0
number_of_responding_apparatus,1.0,167.0,6.0,436.0
number_of_responding_personnel,4.0,515.0,21.0,1275.0
property_use,"896 - Sidewalk, street, roadway, highway, hwy ...",134 - School - Secondary/Senior High (Gr. 9+),323 - Multi-Unit Dwelling - Over 12 Units,"760 - Sto: Warehouse, variety of items, not cl..."
intersection,Bay St / Lake Shore Blvd W,Eglinton Ave W / Yarrow Rd,Beecroft Rd / Bogert Ave / Albert Standing Par...,Pelham Ave / Osler St


check decimal values in count columns

In [19]:
count_columns = [
    "civilian_casualties",
    "tfs_firefighter_casualties",
    "count_of_persons_rescued",
    "number_of_responding_apparatus",
    "number_of_responding_personnel",
    "estimated_number_of_persons_displaced"
]

for col in count_columns:
    values = df_clean[col].dropna()

    decimal_count = (values % 1 != 0).sum()

    print(f"{col}: {decimal_count} decimal values")

civilian_casualties: 0 decimal values
tfs_firefighter_casualties: 0 decimal values
count_of_persons_rescued: 0 decimal values
number_of_responding_apparatus: 0 decimal values
number_of_responding_personnel: 0 decimal values
estimated_number_of_persons_displaced: 0 decimal values


final data quality summary

In [20]:
print("FINAL DATA CLEANING SUMMARY")
print("-" * 40)

print("Total rows:", df_clean.shape[0])
print("Total columns:", df_clean.shape[1])

print("\nDuplicate rows:", df_clean.duplicated().sum())

print(
    "Duplicate incident numbers:",
    df_clean["incident_number"].duplicated().sum()
)

print("\nMissing values by column:")
display(
    df_clean.isna().sum()[
        df_clean.isna().sum() > 0
    ].sort_values(ascending=False)
)

print("\nData types:")
display(df_clean.dtypes)

FINAL DATA CLEANING SUMMARY
----------------------------------------
Total rows: 36564
Total columns: 43

Duplicate rows: 0
Duplicate incident numbers: 0

Missing values by column:


exposures                                                        25418
business_impact                                                  18487
smoke_spread                                                     18487
extent_of_fire                                                   18487
sprinkler_system_operation                                       18486
smoke_alarm_at_fire_origin_alarm_type                            18486
smoke_alarm_at_fire_origin_alarm_failure                         18486
smoke_alarm_at_fire_origin                                       18486
fire_alarm_system_operation                                      18486
smoke_alarm_impact_on_persons_evacuating_impact_on_evacuation    18486
sprinkler_system_presence                                        18486
fire_alarm_system_impact_on_evacuation                           18486
level_of_origin                                                  18486
fire_alarm_system_presence                                       18486
estima


Data types:


_id                                                                       int64
incident_number                                                          object
initial_cad_event_type                                                   object
final_incident_type                                                      object
exposures                                                               float64
incident_station_area                                                    object
incident_ward                                                           float64
intersection                                                             object
latitude                                                                float64
longitude                                                               float64
property_use                                                             object
building_status                                                          object
tfs_alarm_time                          

saving the cleaned dataset

In [21]:
from pathlib import Path

# Create the processed folder if it doesn't exist
output_folder = Path("../data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
output_path = output_folder / "fire_incidents_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved to:", output_path)

Cleaned dataset saved to: ..\data\processed\fire_incidents_cleaned.csv
